# elutediff · 02 · RT baselines (the known bar)

Runs on a **free CPU Colab runtime**. Trains the B1/B2 scalar-RT baselines (ECFP+XGBoost/RandomForest, descriptor/ECFP MLP) and split-conformal intervals. These establish the point-MAE bar the DiffusionGemma density model is judged against — a strong graph model may well win on pure MAE, and that's expected.

In [ ]:
# Install elutediff from GitHub (core + chemistry + baselines extras).
%pip install -q "elutediff[chem,baselines] @ git+https://github.com/CodeHalwell/elutediff.git@claude/project-structure-unsloth-bnwe57"
%pip install -q matplotlib

## 1. Targets JSONL

Re-run `01_data_and_targets.ipynb` first, or regenerate a synthetic JSONL here. Set `DATA` to your file from notebook 01 if you saved it.

In [ ]:
import os, json
DATA = '/content/targets.jsonl'
if not os.path.exists(DATA):
    # regenerate a synthetic JSONL inline (same recipe as notebook 01)
    import random
    from rdkit import Chem; from rdkit.Chem import Descriptors
    from elutediff.config import Config
    from elutediff.data.splits import make_split
    from elutediff.targets.density import gaussian_density
    from elutediff.targets.quantize import quantize
    from elutediff.serialization.prompts import build_prompt, target_string
    cfg = Config(); cfg.split.strategy = 'random'; rng = random.Random(0)
    cores = ['c1ccccc1','c1ccncc1','C1CCCCC1','c1ccc2ccccc2c1','C1CCNCC1']
    mols = []
    for i in range(400):
        smi = 'C'*rng.randint(0,8) + rng.choice(cores) + rng.choice(['','O','N','C(=O)O','Cl','OC'])
        m = Chem.MolFromSmiles(smi)
        if m is None: continue
        rt = max(20., min(60 + 80*Descriptors.MolLogP(m) + 0.8*Descriptors.MolWt(m) + rng.uniform(-20,20), 1180.))
        mols.append((Chem.MolToSmiles(m), rt))
    smiles = [s for s,_ in mols]
    split = make_split(smiles, cfg.split)
    fold = {i:n for n,idx in (('train',split.train_idx),('val',split.val_idx),('test',split.test_idx)) for i in idx}
    with open(DATA,'w') as fh:
        for i,(s,rt) in enumerate(mols):
            lv = quantize(gaussian_density(rt, cfg.target), cfg.target)
            fh.write(json.dumps({'smiles':s,'rt':rt,'split':fold.get(i,'train'),
                'prompt':build_prompt(smiles=s,target_cfg=cfg.target,cond_cfg=cfg.conditioning),
                'target':target_string(lv,cfg.target)})+'\n')
rows = [json.loads(l) for l in open(DATA)]
print(len(rows), 'rows | folds:', {k: sum(r['split']==k for r in rows) for k in ('train','val','test')})

## 2. Train & compare B1/B2

Same featurizers and split indices as the rest of the pipeline, so the comparison to the diffusion decoder is apples-to-apples.

In [ ]:
import numpy as np
from elutediff.config import ConditioningConfig
from elutediff.models.baselines import build_baseline, featurize
from elutediff.evaluation.point_rt import point_rt_metrics, tolerance_hit_rate

descs = ConditioningConfig().descriptors
def fold(name): r=[x for x in rows if x['split']==name]; return [x['smiles'] for x in r], np.array([x['rt'] for x in r])
tr_s, tr_y = fold('train'); te_s, te_y = fold('test')

configs = [('rf','ecfp'), ('xgb','ecfp+descriptors'), ('mlp','descriptors')]
for name, feats in configs:
    Xtr = featurize(tr_s, feats, descriptors=descs)
    Xte = featurize(te_s, feats, descriptors=descs)
    model = build_baseline(name).fit(Xtr, tr_y)
    pred = model.predict(Xte)
    m = point_rt_metrics(te_y, pred)
    hits = tolerance_hit_rate(te_y, pred, [15,30,60])
    print(f"{name:>3} | {feats:<18} MAE {m['mae']:6.1f}s  R2 {m['r2']:.3f}  " +
          ' '.join(f'{k} {v*100:.0f}%' for k,v in hits.items()))

## 3. Uncertainty baseline: split-conformal intervals (B9)

Distribution-free 90% intervals calibrated on the val fold. The DiffusionGemma sampler's uncertainty must beat cheap baselines like this to earn an uncertainty claim — otherwise we report a clean negative.

In [ ]:
from elutediff.models.baselines import ConformalInterval
from elutediff.evaluation.uncertainty import interval_coverage, median_interval_width

va_s, va_y = fold('val')
Xtr = featurize(tr_s, 'ecfp+descriptors', descriptors=descs)
Xva = featurize(va_s, 'ecfp+descriptors', descriptors=descs)
Xte = featurize(te_s, 'ecfp+descriptors', descriptors=descs)
base = build_baseline('xgb').fit(Xtr, tr_y)
conf = ConformalInterval(base, level=0.9).calibrate(Xva, va_y)
lo, hi = conf.predict_interval(Xte)
print(f'90% conformal: coverage {interval_coverage(te_y, lo, hi)*100:.1f}%  median width {median_interval_width(lo, hi):.1f}s')